In [ ]:
import requests
import pandas as pd
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import json
from datetime import datetime, timedelta
import os

# Caminho do arquivo de data/hora de última execução
last_update_file = r'G:/Drives compartilhados/Bases BI/data_atualizacao_preco_limite.csv'

# Obter a data e hora atual
data_hora_atual = datetime.now() 

# Verificar se o arquivo de última execução existe
if os.path.exists(last_update_file):
    # Ler o arquivo CSV para obter a última data/hora
    df_last_update = pd.read_csv(last_update_file)
    ultima_data_hora = pd.to_datetime(df_last_update['DataHora'].iloc[0])

    # Calcular a diferença de tempo
    diferenca_tempo = data_hora_atual - ultima_data_hora

    # Verificar se a diferença é menor que 10 minutos
    if diferenca_tempo < timedelta(minutes=30):
        print("O script não será executado, pois a última execução foi há menos de 10 minutos.")
        exit()

# Atualizar o arquivo de última execução
df_data_hora = pd.DataFrame({'DataHora': [data_hora_atual]})
df_data_hora.to_csv(last_update_file, index=False)

# Headers para a API VTEX
headers = {
    'Accept': "application/json",
    'Content-Type': "application/json",
    'X-VTEX-API-AppKey': "vtexappkey-epocacosmeticos-DRCNSK",
    'X-VTEX-API-AppToken': "TJNIBPASJIKRGZFVFMUCVSDZTBKPWOVJKLOONIRDERWILXWABNTPXUKOWOHOMAYUYLHIYAXSQXVSXOVCLKDVUHTDAYRZLKZVUYVYGODIZOPMDRGLOAUTMVIEDPNOSPJZ"
}

# Caminho do arquivo CSV
csv_file_path = r'G:/Drives compartilhados/Bases BI/preco_limite.csv'
print("Leu arquivo")
df_1 = pd.read_csv(csv_file_path, sep=';')
skus = df_1['idSku']
items = [] 

# Loop para realizar as requisições
for idSku in skus:
    # Requisição de preços
    url_price = f"https://epocacosmeticos.vtexcommercestable.com.br/api/pricing/prices/{idSku}"
    response_price = requests.get(url_price, headers=headers)
    
    # Requisição de informações do produto
    url_sku = f"https://epocacosmeticos.vtexcommercestable.com.br/api/catalog_system/pvt/sku/stockkeepingunitbyid/{idSku}"
    response_sku = requests.get(url_sku, headers=headers)

    # Verificação das respostas
    if response_price.status_code != 200 or 'not found' in response_price.text:
        valor = 0
        limite = 0
    else:
        valor = response_price.json().get('basePrice', 0)
        limite = df_1[df_1['idSku'] == idSku]['Limite'].values[0]
        limite = round(float(limite.replace(',', '.')), 2)

    if response_sku.status_code != 200 or 'Error' in response_sku.text:
        descricao = ''
    else:
        descricao = response_sku.json().get('NameComplete', '')

    items.append({
        'ID_SKU': idSku, 
        'Descrição': descricao, 
        'Preço Praticado': float(valor), 
        'Preço Limite': float(limite)
    })

# Criação do dataframe e exportação do CSV
df = pd.DataFrame(items)
df_filtro = df[df['Preço Limite'] > df['Preço Praticado']]
path = r'G:/Drives compartilhados/Bases BI/alerta_preco.csv'
df_filtro.to_csv(path, index=False)
print('Base criada.')

# --- Função de envio para o Google Chat ---
def enviar_mensagem_gchat(url_webhook, mensagem):
    """
    Envia uma mensagem simples para um espaço do Google Chat via Webhook.
    """
    headers = {'Content-Type': 'application/json; charset=UTF-8'}
    payload = {'text': mensagem}
    try:
        resposta = requests.post(url_webhook, headers=headers, data=json.dumps(payload))
        resposta.raise_for_status()
        print("Mensagem para o Google Chat enviada com sucesso!")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao enviar mensagem para o Google Chat: {e}")

# --- Envio de alertas ---
if not df_filtro.empty:
    # --- Envio de E-mail ---
    print("Iniciando envio de e-mails...")
    emails_mandar = [
        "renan@epocacosmeticos.com.br", "mariana.mattos@epocacosmeticos.com.br", "paula@epocacosmeticos.com.br",
        "gabriellasantos@magazineluiza.com.br", "mariana.rivas@epocacosmeticos.com.br", "dafne@epocacosmeticos.com.br",
        "daiane@epocacosmeticos.com.br", "maria.moreira@epocacosmeticos.com.br", "natalia.rodrigues@epocacosmeticos.com.br",
        "c.ossaille@magazineluiza.com.br", "erika.ribeiro@epocacosmeticos.com.br", "debora.vignoli@epocacosmeticos.com.br",
        "tulani.dias@epocacosmeticos.com.br", "maria.amanda@epocacosmeticos.com.br", "ramos.filipe@magazineluiza.com.br",
        "matos.mariana@epocacosmeticos.com.br", "jesus.thiago@epocacosmeticos.com.br", "marcelo.scosta@epocacosmeticos.com.br",
        "juan.carvalho@epocacosmeticos.com.br", "marianna.mendes@epocacosmeticos.com.br"
    ]
    
    smtp_server = "smtp.gmail.com"
    smtp_port = 587
    sender_email = "joao.pcarvalho@magazineluiza.com.br"
    app_password = "ohci bjqd bkky vkbf"
    
    html = df_filtro.to_html(classes='table', index=False)
    html = html.replace('<th>', '<th style="text-align: center;">').replace('<td>', '<td style="text-align: center;">')
    subject = "[URGENTE] Alerta Preço"
    body = f"{html}"

    msg = MIMEMultipart()
    msg["From"] = sender_email
    msg["To"] = ", ".join(emails_mandar) # Envia para todos de uma vez
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "html"))

    with smtplib.SMTP(smtp_server, smtp_port) as server:
        server.starttls()
        server.login(sender_email, app_password)
        server.sendmail(sender_email, emails_mandar, msg.as_string())
    print("Email enviado!")

    # --- Envio para o Google Chat ---
    print("Iniciando envio para o Google Chat...")
    WEBHOOK_URL = "Inserir a sua webhook aqui"
    # Constrói a mensagem iterando sobre o DataFrame
    mensagem_partes = ["🔥🔥*ATENÇÃO ALERTA DE PREÇO!!!*🔥🔥\n"]
    for index, row in df_filtro.iterrows():
        mensagem_partes.append(
            f"Produto: {row['ID_SKU']} - {row['Descrição']}\n"
            f"Preço Praticado: {row['Preço Praticado']:.2f}\n"
            f"Preço Limite: {row['Preço Limite']:.2f}\n"
        )
    
    mensagem_partes.append("<users/all>")
    MENSAGEM = "\n".join(mensagem_partes)
    
    enviar_mensagem_gchat(WEBHOOK_URL, MENSAGEM)

else:
    print("Nenhum alerta de preço encontrado.")